# Kodra AI Agent: Cloud/Colab GPU Training Preparation Notebook

**Product:** Kodra AI Agent
**Model:** Kodra GPT (`KodraGPT`)
**Core:** Kodra Core
**Tagline:** CODE - THINK - CREATE

This notebook prepares and validates a GPU training run for **Kodra GPT** on Google Colab (or any
Jupyter environment with a CUDA GPU). Every cell is labeled `CELL NN` and runs in a fixed,
sequential order: clone -> install -> verify hardware -> build/validate a clean dataset -> verify
no metadata contamination -> tokenizer -> model selection -> dataloaders -> causal-shift proof ->
a 20-step GPU smoke test -> checkpoint save/reload/resume -> generation/syntax evaluation ->
optional Drive backup. Full multi-epoch training lives behind an explicit **RUN MANUALLY ONLY**
gate in the final cell and never runs as part of "Run All".

### Why earlier runs produced malformed Python and a literal `target:` in generated text

A prior diagnostic pass flagged the substring `target:` inside the training corpus as "leaked
metadata" (as if it were a serialized `{"target": ...}` record field). That diagnosis was a
**false positive**: the only place `target:` appears in the corpus is `data/code/algorithms.py`'s
binary-search implementation - `target` is an ordinary Python parameter name, and the `:` is the
block colon of `if arr[mid] == target:` / `elif arr[mid] < target:`. That is completely valid,
intentional Python, not contamination. `CELL 07` below proves this with a validator that
distinguishes real (quoted-key) metadata leakage from legitimate code, and both this notebook and
`datasets/corpus_pipeline.py` now reject the former while never flagging the latter.

The real reason generated Python was malformed and 0% syntax-parse rate was observed is plain
**severe undertraining**: a ~700-1200 character corpus, a BPE vocabulary that only reached ~415
tokens against a target of 8000, and 20-50 optimizer steps are nowhere near enough for a model to
learn Python syntax. `def quicksort(arr):target:` is the model regurgitating a memorized fragment
of the binary-search function it saw a handful of times, glued onto a different prompt - a
symptom of too little data and too few steps, not a data-pipeline bug. No further serious training
should happen until a larger, validated corpus (`CELL 06`) is used, per Phase F/H.

No credentials are embedded in this notebook. If you want to persist checkpoints to Google Drive,
mount it yourself in Colab and pass that path as `CHECKPOINT_DIR` below.

In [ ]:
# CELL 01 - Clone the repository (HTTPS, works unauthenticated on Colab) and enter kodra-core
!rm -rf /content/Kodra-ai
!git clone https://github.com/ChaceEthan/Kodra-ai.git /content/Kodra-ai
%cd /content/Kodra-ai/kodra-core

In [ ]:
# CELL 02 - Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# CELL 03 - Verify CUDA is available
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
print('CUDA available:', CUDA_AVAILABLE)
if not CUDA_AVAILABLE:
    print('No GPU detected - training will fall back to CPU (slow for anything above kodra-tiny).')

In [ ]:
# CELL 04 - Print GPU name and VRAM
if CUDA_AVAILABLE:
    print('Device name:', torch.cuda.get_device_name(0))
    print('Device count:', torch.cuda.device_count())
    print('Total memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('Skipping GPU info - no CUDA device detected.')

In [ ]:
# CELL 05 - (Optional) Mount Google Drive for persistent checkpoint storage.
# Uncomment if you want checkpoints to survive a Colab session restart.
# from google.colab import drive
# drive.mount('/content/drive')
# CHECKPOINT_DIR = '/content/drive/MyDrive/kodra_checkpoints'
CHECKPOINT_DIR = 'checkpoints'
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)

In [ ]:
# CELL 06 - Build/validate the approved dataset.
# `build_manifest` only ever reads a local, explicitly-approved directory (never the internet or
# an arbitrary repo) and rejects bad records before they can reach training: serialized metadata
# (quoted "target": / "prompt": keys), U+FFFD corruption, and minified/generated code - see
# DEFAULT_QUALITY_FILTERS in datasets/corpus_pipeline.py. To scale beyond the bundled sample
# corpus, run the same pipeline from the command line against your own approved directory:
#   python scripts/prepare_training_corpus.py --source <approved-directory> --output data/manifest.json
# and re-point SOURCE_DIR / MANIFEST_PATH below at that directory/manifest - no code changes needed.
from datasets.corpus_pipeline import build_manifest, write_manifest, build_training_text

SOURCE_DIR = 'data/code'  # bundled, repo-owned sample corpus; point at your own approved tree to scale up
MANIFEST_PATH = 'data/manifest.json'

manifest = build_manifest(SOURCE_DIR, seed=42, val_ratio=0.1, test_ratio=0.0,
                           license='project-sample', source='kodra-sample-corpus')
write_manifest(manifest, MANIFEST_PATH)

# TRAIN_TEXT_CANDIDATE is assembled ONLY from the manifest's clean file contents
# (build_training_text re-reads each file fresh from disk) - manifest bookkeeping fields
# (license, source, sha256, ...) never enter the text the model actually trains on.
TRAIN_TEXT_CANDIDATE = build_training_text(manifest, split='train')

print(f'Discovered {manifest.num_files} files, {manifest.total_chars} chars, '
      f'{manifest.total_token_estimate} approx tokens')
print(f'Languages: {manifest.language_counts}')
print(f'Train/val files: {manifest.split_counts["train"]}/{manifest.split_counts["val"]}')
print(f'Rejected - duplicates: {manifest.num_duplicates_removed}, '
      f'encoding: {manifest.num_encoding_rejected}, '
      f'secrets: {manifest.num_secrets_redacted}, '
      f'quality/contamination: {manifest.num_filtered_out} {manifest.filtered_reasons}')

MIN_CHARS_FOR_MEANINGFUL_TRAINING = 50_000
if manifest.total_chars < MIN_CHARS_FOR_MEANINGFUL_TRAINING:
    print(f'\nWARNING: only {manifest.total_chars} chars - this is a smoke-test-sized corpus, '
          f'not enough to teach real Python syntax. Use scripts/prepare_training_corpus.py '
          f'against a larger approved directory before any serious training run.')

In [ ]:
# CELL 07 - Verify no metadata contamination.
# This replaces the old naive `'target:' in TRAIN_TEXT` substring check, which produced a false
# positive on legitimate code (see the notebook intro). The real check requires a QUOTED key -
# "target": / 'prompt': - which is what actual JSON/dict serialization looks like, and is what
# datasets/corpus_pipeline.py now rejects at the source via contains_serialized_metadata().
# Re-validated here on the assembled TRAIN_TEXT_CANDIDATE from CELL 06 (not just per-file, since
# CELL 06 already rejected bad files) as a second, independent safety net before tokenizer training.
from datasets.corpus_pipeline import contains_serialized_metadata, contains_replacement_character

print('--- Metadata contamination check ---')
print('Contains serialized metadata (quoted "target":/"prompt":/etc key)?',
      contains_serialized_metadata(TRAIN_TEXT_CANDIDATE))
print('Contains U+FFFD replacement character?',
      contains_replacement_character(TRAIN_TEXT_CANDIDATE))

# The bare substring is expected to appear (binary_search's `target` parameter) and is fine -
# only a QUOTED key would be a real problem.
print("Bare 'target:' substring present (expected, from binary_search - not contamination):",
      'target:' in TRAIN_TEXT_CANDIDATE)

assert not contains_serialized_metadata(TRAIN_TEXT_CANDIDATE), 'Real metadata contamination detected - fix the source before training.'
assert not contains_replacement_character(TRAIN_TEXT_CANDIDATE), 'Corpus contains U+FFFD - fix the source before training.'
print('\nOK: no real metadata contamination, no replacement-character corruption.')

In [ ]:
# CELL 08 - Train (or load) the tokenizer. BPE is the default for GPU runs; the Phase 1 char
# tokenizer remains available for parity/debug comparisons.
import os
from tokenizer.char_tokenizer import CharTokenizer
from tokenizer.bpe_tokenizer import ByteLevelBPETokenizer

USE_BPE = True  # set False to use the Phase 1 char tokenizer instead
BPE_VOCAB_PATH = 'tokenizer/vocab_bpe.json'
CHAR_VOCAB_PATH = 'tokenizer/vocab.json'

if USE_BPE:
    tokenizer = ByteLevelBPETokenizer(vocab_size=8000)
    if os.path.exists(BPE_VOCAB_PATH):
        tokenizer.load(BPE_VOCAB_PATH)
        print(f'Loaded existing BPE tokenizer from {BPE_VOCAB_PATH}')
    else:
        tokenizer.train(TRAIN_TEXT_CANDIDATE)
        tokenizer.save(BPE_VOCAB_PATH)
        print(f'Trained a new BPE tokenizer and saved it to {BPE_VOCAB_PATH}')
else:
    tokenizer = CharTokenizer()
    if os.path.exists(CHAR_VOCAB_PATH):
        tokenizer.load(CHAR_VOCAB_PATH)
        print(f'Loaded existing char tokenizer from {CHAR_VOCAB_PATH}')
    else:
        tokenizer.train(TRAIN_TEXT_CANDIDATE)
        tokenizer.save(CHAR_VOCAB_PATH)
        print(f'Trained a new char tokenizer and saved it to {CHAR_VOCAB_PATH}')

print('Tokenizer type:', tokenizer.tokenizer_type, '| vocab size:', tokenizer.vocab_size)

In [ ]:
# CELL 09 - Tokenizer round-trip test.
# Must preserve Python whitespace/newlines exactly - this is what the model actually trains on.
python_sample = (
    "def quicksort(arr):\n"
    "    if len(arr) <= 1:\n"
    "        return arr\n"
    "\n"
    "    pivot = arr[len(arr) // 2]\n"
)
encoded = tokenizer.encode(python_sample)
decoded = tokenizer.decode(encoded)

print(f'Original:\n{python_sample!r}')
print(f'Decoded:\n{decoded!r}')
print(f'Match: {python_sample == decoded}')
assert python_sample == decoded, 'Tokenizer round-trip failed - do not proceed to training.'

In [ ]:
# CELL 10 - Select a Kodra model configuration (kodra-tiny or kodra-small)
from configs.model_sizes import get_model_size, validate_model_config, estimate_resources
from model.gpt_model import KodraGPT

MODEL_SIZE = 'kodra-tiny'  # one of: kodra-tiny, kodra-small (kodra-base/kodra-medium are roadmap-only)
spec = get_model_size(MODEL_SIZE)
model_cfg = spec.config
model_cfg.vocab_size = tokenizer.vocab_size
validate_model_config(model_cfg)

device = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')
model = KodraGPT(model_cfg).to(device)
print(f'Selected {spec.display_name} on {device}')

In [ ]:
# CELL 11 - Exact parameter count (always from the instantiated model, never the roadmap estimate)
from configs.model_sizes import estimate_resources

param_count = model.count_parameters()
resource_estimate = estimate_resources(model_cfg)
print(f'{spec.display_name}: {param_count:,} exact parameters ({param_count/1e6:.2f}M) on {device}')
print(f'Previously trained in this repo: {spec.trained}')
print(f'Rough planning estimate: {resource_estimate["parameters"]:,} params | '
      f'training_vram~{resource_estimate["training_vram_gb"]:.2f}GB | '
      f'inference_vram~{resource_estimate["inference_vram_gb"]:.2f}GB')

In [ ]:
# CELL 12 - Build train/validation dataloaders and the trainer
from configs.config import TrainingConfig
from datasets.dataset import create_dataloader
from training.trainer import Trainer
from training.utils import set_seed

set_seed(42)

# Deterministic held-out split of the corpus for a train/validation loss signal.
_split_idx = int(len(TRAIN_TEXT_CANDIDATE) * 0.9)
TRAIN_TEXT = TRAIN_TEXT_CANDIDATE[:_split_idx]
VAL_TEXT = TRAIN_TEXT_CANDIDATE[_split_idx:]

train_cfg = TrainingConfig(batch_size=16, learning_rate=3e-4, max_epochs=10)
train_loader = create_dataloader(TRAIN_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size)
val_loader = create_dataloader(VAL_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size, shuffle=False)

dataset_manifest_id = f'{manifest.source}-seed{manifest.seed}-{manifest.created_at}'

trainer = Trainer(
    model, train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)
print(f'train batches: {len(train_loader)} | val batches: {len(val_loader)}')

In [ ]:
# CELL 13 - Verify causal next-token shift.
# CodeDataset.__getitem__ must produce input_ids = tokens[:-1], target_ids = tokens[1:] (see
# datasets/dataset.py). Proven directly here, not just asserted in prose.
batch_iter = iter(train_loader)
inputs, targets = next(batch_iter)

sample_tokens = tokenizer.encode(TRAIN_TEXT)[: model_cfg.context_length + 1]
expected_x = sample_tokens[:-1]
expected_y = sample_tokens[1:]

print('First batch input[0][:10]: ', inputs[0][:10].tolist())
print('First batch target[0][:10]:', targets[0][:10].tolist())
print('Expected input[:10] from raw tokens: ', expected_x[:10])
print('Expected target[:10] from raw tokens:', expected_y[:10])

assert inputs[0].tolist() == expected_x, 'Causal shift broken: input_ids != tokens[:-1]'
assert targets[0].tolist() == expected_y, 'Causal shift broken: target_ids != tokens[1:]'
assert inputs[0].tolist()[1:] == targets[0].tolist()[:-1], 'target is not input shifted by exactly one token'
print('\nOK: causal next-token shift verified (input_ids = tokens[:-1], target_ids = tokens[1:]).')

In [ ]:
# CELL 14 - 20-step GPU smoke training: validates the pipeline end-to-end (forward/backward/step,
# AMP, checkpointing) before committing to a full run. This is NOT the full training loop.
SMOKE_TRAIN_STEPS = 20

smoke_epoch = 0
while trainer.step_count < SMOKE_TRAIN_STEPS:
    smoke_epoch += 1
    smoke_avg_loss = trainer.train_epoch(smoke_epoch, total_steps=SMOKE_TRAIN_STEPS)
    print(f'[smoke] epoch {smoke_epoch} | avg_loss={smoke_avg_loss:.4f} | step={trainer.step_count}')

print(f'Smoke training complete at step {trainer.step_count} (target was {SMOKE_TRAIN_STEPS}).')

In [ ]:
# CELL 15 - Train/validation loss after the smoke run
smoke_val_loss = trainer.evaluate()
print(f'Train loss (last step): {trainer.history[-1]["loss"]:.4f}')
print(f'Validation loss: {smoke_val_loss}')

In [ ]:
# CELL 16 - Save checkpoint
trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=smoke_val_loss)
print(f'Saved checkpoint to {os.path.join(CHECKPOINT_DIR, "kodra_gpt_latest.pt")}')

In [ ]:
# CELL 17 - Reload checkpoint into a fresh trainer/model instance
reload_model = KodraGPT(model_cfg).to(device)
reload_trainer = Trainer(
    reload_model, train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)

latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'kodra_gpt_latest.pt')
reload_trainer.load_checkpoint(latest_checkpoint_path)
print(f'Reloaded trainer from checkpoint: {latest_checkpoint_path}')
print(f'Reloaded step_count: {reload_trainer.step_count} (expected {trainer.step_count})')
assert reload_trainer.step_count == trainer.step_count, 'Checkpoint reload lost step_count.'

In [ ]:
# CELL 18 - Resume training for exactly 5 more steps on the reloaded trainer
RESUME_STEPS = 5
resume_target = reload_trainer.step_count + RESUME_STEPS
resume_epoch = 0
while reload_trainer.step_count < resume_target:
    resume_epoch += 1
    resume_avg_loss = reload_trainer.train_epoch(resume_epoch, total_steps=resume_target)
    print(f'[resume] epoch {resume_epoch} | avg_loss={resume_avg_loss:.4f} | step={reload_trainer.step_count}')

print(f'Resumed training to step {reload_trainer.step_count} (target was {resume_target}).')
assert reload_trainer.step_count == resume_target, 'Resume did not advance by exactly RESUME_STEPS.'

In [ ]:
# CELL 19 - Evaluate generation/syntax.
# CodeGenerator.generate() returns prompt + continuation (documented contract, see
# inference/generator.py); python_parses() is called on that exact string with no
# post-processing, so the syntax score is never artificially inflated. A low/0% parse rate here
# on a smoke-trained model is expected (see the notebook intro) - it is a true reflection of an
# undertrained model, not an evaluator bug.
from evaluation.evaluator import full_evaluation_report
from inference.generator import CodeGenerator
import json as _json

report = full_evaluation_report(reload_model, tokenizer, device, val_loader)
print(_json.dumps(report, indent=2, default=str))

generator = CodeGenerator(reload_model, tokenizer, device)
sample = generator.generate('def quicksort(arr):', max_new_tokens=64, temperature=0.7, top_k=40)
print('\nSample completion (raw generator output, prompt + continuation):')
print(repr(sample))

In [ ]:
# CELL 20 - (Optional) Back up checkpoints to Google Drive. Only runs if Drive is mounted (see
# the optional cell above) and BACKUP_TO_DRIVE is set True.
BACKUP_TO_DRIVE = False
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/kodra_checkpoints_backup'

if BACKUP_TO_DRIVE:
    import shutil
    if not os.path.isdir('/content/drive'):
        print('Google Drive is not mounted - skipping backup. Mount it in CELL 05 first.')
    else:
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        for fname in os.listdir(CHECKPOINT_DIR):
            shutil.copy2(os.path.join(CHECKPOINT_DIR, fname), os.path.join(DRIVE_BACKUP_DIR, fname))
        print(f'Backed up checkpoints from {CHECKPOINT_DIR} to {DRIVE_BACKUP_DIR}')
else:
    print('BACKUP_TO_DRIVE is False - skipping Drive backup.')

In [ ]:
# CELL 21 - FULL TRAINING - RUN MANUALLY ONLY
#
# Everything above only validated the pipeline (20-step smoke test + 5-step resume) on the small
# bundled sample corpus. Per the root-cause diagnosis in this notebook's intro, the checkpoint
# produced by CELLS 14-18 is a diagnostic/legacy checkpoint only - it was never intended as a
# production starting point and should NOT be used as one. Do not flip RUN_FULL_TRAINING until:
#   1. CELL 06 has been pointed at a larger, approved corpus (data/code's ~700 chars is a smoke-test
#      fixture, not a training set), and
#   2. CELLS 07/09/13 above pass on that corpus.
# Full training is a long-running, resource-consuming operation and must be started deliberately;
# RUN_FULL_TRAINING defaults to False so re-running the whole notebook (e.g. "Run All") never
# triggers a full run.
RUN_FULL_TRAINING = False

if not RUN_FULL_TRAINING:
    raise RuntimeError(
        'Full training is gated. Set RUN_FULL_TRAINING = True above and re-run this '
        'cell to start the full training loop from scratch on a validated corpus.'
    )

total_steps = train_cfg.max_epochs * len(train_loader)
for epoch in range(1, train_cfg.max_epochs + 1):
    avg_loss = trainer.train_epoch(epoch, total_steps=total_steps)
    val_loss = trainer.evaluate()
    trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=val_loss)
    tps = trainer.history[-1]['tokens_per_sec'] if trainer.history else 0.0
    print(f'Epoch {epoch}/{train_cfg.max_epochs} | avg_loss={avg_loss:.4f} | '
          f'val_loss={val_loss} | step={trainer.step_count} | tokens/sec={tps:.0f}')